In [1]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

PROJECT_RANDOM_STATE = 2627


# Practice 3 — Evaluation and comparison (R4)

**Scope:** canonical folds, shared metric code, `comparativa.csv` and the paired per-fold differences of P3.5.

## Setup

In [2]:
# Standard library: read the JSON file with the class names
import json

# Numerical arrays and tables
import numpy as np
import pandas as pd

# scikit-learn: baselines, metrics and out-of-fold predictions
import sklearn
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, mean_absolute_error,
                             mean_squared_error, precision_recall_fscore_support, r2_score)
from sklearn.model_selection import cross_val_predict

# Library versions: some results can change from one version to another
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)

numpy        2.4.6
pandas       3.0.3
scikit-learn 1.9.0


### Shared helpers (used by both challenges)

These functions are used in the cyber and in the storage part, so this cell goes into
both final notebooks.

- `canonical_cv` reads the course fold file and returns `folds`, the list of
  `(train_idx, val_idx)` pairs. The folds are read, never rebuilt. As in Practice 2, all the
  folds go through this one function.
- `check_folds` checks the grouping rule: in every fold, no group is both in training and in
  validation, and every row is validated exactly once.
- `mean_and_spread` gives the mean and the spread (max − min) of the five fold values, the
  same definition used in `comparativa.csv`.

In [3]:
# K, the number of canonical folds
CV_FOLDS = 5


# Read the canonical folds of one challenge ("cyber" or "storage") from the course file
def canonical_cv(challenge, dev):
    """Read the course fold file and return the folds as (train_idx, val_idx) pairs."""
    # The course file has two columns: row_id and fold_id (0 to 4)
    fold_table = pd.read_csv("../folds/" + challenge + "_folds.csv")

    # The positions only make sense if both files list the rows in the same order
    same_order = np.array_equal(fold_table["row_id"].to_numpy(), dev["row_id"].to_numpy())
    assert same_order, "the fold file and the data table list the rows in a different order"

    # Fold number of each row, as a NumPy array
    fold_id = fold_table["fold_id"].to_numpy()

    # One (train_idx, val_idx) pair per fold
    folds = []
    for f in range(CV_FOLDS):
        train_idx = np.where(fold_id != f)[0]   # positions of the rows of the other four folds
        val_idx = np.where(fold_id == f)[0]     # positions of the rows of fold f
        folds.append((train_idx, val_idx))
    return folds

In [4]:
# Check the grouping rule of the canonical folds and print the size of each fold
def check_folds(folds, groups):
    """Check that no group is split between training and validation, fold by fold."""
    # Group of each row (feature_group in cyber, config_group in storage)
    groups = np.asarray(groups)
    # How many times each row appears in a validation part; it must end at 1
    times_validated = np.zeros(len(groups), dtype=int)

    for number, (train_idx, val_idx) in enumerate(folds, start=1):
        # Groups that appear both in the training part and in the validation part
        shared = set(groups[train_idx]) & set(groups[val_idx])
        print(f"fold {number}: {len(train_idx)} training rows, {len(val_idx)} validation rows, "
              f"{len(shared)} groups in both parts")
        # Stop the notebook if any group is split between the two parts
        assert len(shared) == 0, "a group is in training and in validation"
        # Count one more validation for the rows of this fold
        times_validated[val_idx] = times_validated[val_idx] + 1

    # Every row must be in the validation part of exactly one fold
    assert (times_validated == 1).all(), "a row is not validated exactly once"
    print("every row is validated exactly once")


# Summary of five per-fold values, with the same definition of spread as comparativa.csv
def mean_and_spread(values):
    """Mean and spread (max - min) of the five fold values."""
    values = np.asarray(values, dtype=float)
    return values.mean(), values.max() - values.min()

### Helpers for `comparativa.csv` and the paired differences

- `append_comparativa_row` adds one row at the end of the table. It never rewrites the rows
  that are already there, and it does not add the same row twice if the notebook is run again.
  It is based on the function of R4 in Practice 2.
- `table_scores` reads the five per-fold values of one row of the table.
- `paired_differences` compares two rows fold by fold. The sign follows the statement
  (Practice 2, Section 5.3): a positive difference always means that the new row did better
  on that fold. For macro F1, $\delta_f = \text{new}_f - \text{P2}_f$; for MAE(z), where lower
  is better, $\delta_f = \text{P2}_f - \text{new}_f$.

The differences are computed from the values as written in the table (four decimals), so
anyone can recompute them from `comparativa.csv`.

In [5]:
# The comparison table, in the root of the repository
COMPARISON_PATH = "../comparativa.csv"


# Turn five numbers into the text format of comparativa.csv, e.g. "0.9436 0.9473 ..."
def fold_text(values):
    """Write the five fold values as one text with 4 decimals, as in comparativa.csv."""
    parts = []
    for value in values:
        parts.append(f"{value:.4f}")   # 4 decimals, e.g. 0.9436
    return " ".join(parts)             # joined with spaces


# Add one row at the end of comparativa.csv without touching the rows already there
def append_comparativa_row(row):
    """Add one row at the end of comparativa.csv. The rows already there are never changed."""
    table = pd.read_csv(COMPARISON_PATH)

    # 1. The new row must have exactly the columns of the table, in the same order
    assert list(row.keys()) == list(table.columns), "the row does not have the table columns"

    # 2. If the same row is already in the table (notebook run again), do not add it twice
    same_row = ((table["milestone"] == row["milestone"])
                & (table["challenge"] == row["challenge"])
                & (table["family"] == row["family"])
                & (table["route"] == row["route"]))
    if same_row.any():
        print("already in comparativa.csv:", row["milestone"], row["challenge"], row["family"])
        return

    # 3. mode="a" writes at the end of the file and leaves every earlier line as it was
    pd.DataFrame([row]).to_csv(COMPARISON_PATH, mode="a", header=False, index=False)
    print("added to comparativa.csv:", row["milestone"], row["challenge"], row["family"])

In [6]:
# Read the five per-fold values of the primary metric of one row of comparativa.csv
def table_scores(milestone, challenge, family):
    """Five per-fold values of the primary metric of one row of comparativa.csv."""
    table = pd.read_csv(COMPARISON_PATH)
    # True only for the row with this milestone, challenge and family
    is_row = ((table["milestone"] == milestone)
              & (table["challenge"] == challenge)
              & (table["family"] == family))
    # Stop the notebook if there is no such row, or more than one
    assert is_row.sum() == 1, "expected exactly one row for " + milestone + " " + family
    # The values are stored as one text, e.g. "0.9436 0.9473 ..."
    text = table.loc[is_row, "primary_metric_per_fold"].iloc[0]
    # Split the text at the spaces and turn each part into a number
    return np.array(text.split(), dtype=float)


# Compare a new row with a Practice 2 row fold by fold and print a short summary
def paired_differences(challenge, new_scores, old_scores):
    """delta for each fold; a positive delta always means the new row did better."""
    if challenge == "cyber":
        delta = new_scores - old_scores    # macro F1: higher is better
    else:
        delta = old_scores - new_scores    # MAE(z): lower is better

    # One row per fold: Practice 2 value, new value and their difference
    table = pd.DataFrame({"P2": old_scores, "new": new_scores, "delta": delta},
                         index=range(1, CV_FOLDS + 1))
    table.index.name = "fold"
    print(table.round(4).to_string())

    # Descriptive summary of the five differences (not a statistical test)
    mean_delta, spread_delta = mean_and_spread(delta)
    print(f"mean delta            {mean_delta:+.4f}")
    print(f"spread of delta       {spread_delta:.4f}   (max - min)")
    print(f"folds with delta > 0  {(delta > 0).sum()} of {CV_FOLDS}")
    # Fold-to-fold variability of each row, to judge the size of the change
    print(f"spread of P2 row      {mean_and_spread(old_scores)[1]:.4f}")
    print(f"spread of new row     {mean_and_spread(new_scores)[1]:.4f}")
    return table

**Check of the paired-difference code.** We compare two Practice 2 rows whose differences
Practice 2 already printed: KNN against Dummy in Challenge A, with differences 0.9194,
0.9231, 0.9248, 0.9187 and 0.9209. The values must match up to the four-decimal rounding of
the table (on fold 4 the table gives 0.9186).

In [7]:
# Two Practice 2 rows of comparativa.csv
knn_scores = table_scores("P2", "cyber", "KNN")
dummy_scores = table_scores("P2", "cyber", "Dummy")

# KNN plays the role of the "new" row and Dummy the role of the "old" one
check = paired_differences("cyber", knn_scores, dummy_scores)

# Each table value is rounded to 4 decimals, so a difference can move by 0.0001
printed_in_p2 = [0.9194, 0.9231, 0.9248, 0.9187, 0.9209]
assert np.allclose(check["delta"], printed_in_p2, atol=0.00015)
print("\nthe function gives the differences printed in Practice 2")

          P2     new   delta
fold                        
1     0.0242  0.9436  0.9194
2     0.0242  0.9473  0.9231
3     0.0242  0.9490  0.9248
4     0.0242  0.9428  0.9186
5     0.0242  0.9451  0.9209
mean delta            +0.9214
spread of delta       0.0062   (max - min)
folds with delta > 0  5 of 5
spread of P2 row      0.0000
spread of new row     0.0062

the function gives the differences printed in Practice 2


# Challenge A — cyber

> **Only in this draft.** The next cell loads the data with the shared names of the team
> template. In the final notebook these names come from R1's cells, so R5 does not copy it.

In [8]:
# Only for this draft notebook: in p3_cyber.ipynb these names come from R1's cells

# Development table and grouping metadata (the groups never enter the model)
dev = pd.read_csv("../datos/cyber/cyber_development.csv")
groups = pd.read_csv("../datos/cyber/cyber_groups.csv")

# The 11 source predictors: 3 nominal and 8 numeric
NOMINAL = ["proto", "service", "conn_state"]
NUMERIC = ["duration", "src_bytes", "dst_bytes", "missed_bytes",
           "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes"]
FEATURES = NOMINAL + NUMERIC

X = dev[FEATURES]                       # only the 11 source predictors
y = dev["attack_code"].to_numpy()       # class code of each row (0 to 9)

# Class code -> class name, e.g. 7 -> "ransomware"
CLASS_NAMES = {}
with open("../datos/cyber/cyber_class_codes.json") as file:
    for code, name in json.load(file).items():
        CLASS_NAMES[int(code)] = name   # JSON keys are text, so we turn them into integers

## Canonical folds and metrics (publish for R2)

The folds are read from `folds/cyber_folds.csv`, the course file. They come from a
`StratifiedGroupKFold(5)` on `feature_group`, so rows with identical predictors always stay in
the same fold.

In [9]:
# The groups file must list the rows in the same order as the data table
assert np.array_equal(groups["row_id"].to_numpy(), dev["row_id"].to_numpy())

# Canonical folds of Challenge A: a list of five (train_idx, val_idx) pairs
folds = canonical_cv("cyber", dev)

In [10]:
# Grouping rule: no feature_group may be in training and in validation of the same fold
check_folds(folds, groups["feature_group"])

# X must only hold the source predictors: never row_id, feature_group, config_group or fold_id
assert list(X.columns) == FEATURES

fold 1: 27942 training rows, 6986 validation rows, 0 groups in both parts
fold 2: 27942 training rows, 6986 validation rows, 0 groups in both parts
fold 3: 27943 training rows, 6985 validation rows, 0 groups in both parts
fold 4: 27942 training rows, 6986 validation rows, 0 groups in both parts
fold 5: 27943 training rows, 6985 validation rows, 0 groups in both parts
every row is validated exactly once


**Metric code for Challenge A** (the metric contract of Practice 2). The model is fitted
outside these functions, for example with `cross_val_predict(model, X, y, cv=folds)`, which
fits a fresh copy of the whole `Pipeline` on the training rows of each fold and predicts its
validation rows. So every row gets one prediction from a model that did not see it, and the
encoder and the scaler are fitted inside each fold.

- `cyber_fold_metrics`: macro F1 (primary), weighted F1 and accuracy on each fold.
- `cyber_class_table`: precision, recall and F1 of each class, beside its support.
- `cyber_confusion`: confusion matrix with the class names (rows = true, columns = predicted).

In [11]:
# Metrics of Challenge A on each fold, from the out-of-fold predictions
def cyber_fold_metrics(y, y_pred, folds):
    """Macro F1 (primary), weighted F1 and accuracy on the validation rows of each fold."""
    y = np.asarray(y)
    rows = []
    for number, (train_idx, val_idx) in enumerate(folds, start=1):
        # True classes and predicted classes of the validation rows of this fold
        y_true_fold = y[val_idx]
        y_pred_fold = y_pred[val_idx]
        # One dictionary per fold with its three metrics
        rows.append({"fold": number,
                     "macro_f1": f1_score(y_true_fold, y_pred_fold, average="macro"),
                     "weighted_f1": f1_score(y_true_fold, y_pred_fold, average="weighted"),
                     "accuracy": accuracy_score(y_true_fold, y_pred_fold)})
    # Table with one row per fold
    return pd.DataFrame(rows).set_index("fold")

In [12]:
# Class codes in increasing order, and their names in the same order
CLASS_CODES = sorted(CLASS_NAMES)                     # 0, 1, ..., 9
CLASS_LABELS = []
for class_code in CLASS_CODES:
    CLASS_LABELS.append(CLASS_NAMES[class_code])      # "backdoor", "ddos", ...


# Per-class metrics on all the out-of-fold predictions together
def cyber_class_table(y, y_pred):
    """Precision, recall and F1 of each class, beside its support (all folds together)."""
    # One value per class, in the order of CLASS_CODES; support = number of true rows
    precision, recall, f1, support = precision_recall_fscore_support(
        y, y_pred, labels=CLASS_CODES, zero_division=0)
    return pd.DataFrame({"precision": precision, "recall": recall, "f1": f1,
                         "support": support}, index=CLASS_LABELS)


# Confusion matrix that can be read without remembering the class codes
def cyber_confusion(y, y_pred):
    """Confusion matrix with class names: rows = true class, columns = predicted class."""
    matrix = confusion_matrix(y, y_pred, labels=CLASS_CODES)
    table = pd.DataFrame(matrix, index=CLASS_LABELS, columns=CLASS_LABELS)
    table.index.name = "true"
    table.columns.name = "predicted"
    return table

**Check: the folds and the metric code reproduce Practice 2.** We run again the Practice 2
baseline, `DummyClassifier(strategy="most_frequent")`, and compare its macro F1 per fold
with the P2 row of `comparativa.csv`. If the folds or the metric code had changed, the
values would not match. The Dummy model does not use `X`, so it needs no preprocessing.

In [13]:
# Out-of-fold predictions of the Practice 2 baseline on the canonical folds
y_pred_dummy = cross_val_predict(DummyClassifier(strategy="most_frequent"), X, y, cv=folds)

# Its metrics on each fold
dummy_metrics = cyber_fold_metrics(y, y_pred_dummy, folds)
print(dummy_metrics.round(4).to_string())

# They must equal the P2 Dummy row of comparativa.csv (rounded to 4 decimals there)
p2_dummy = table_scores("P2", "cyber", "Dummy")
assert np.allclose(dummy_metrics["macro_f1"], p2_dummy, atol=0.00005)
print("\nmacro F1 per fold equals the P2 Dummy row of comparativa.csv")

      macro_f1  weighted_f1  accuracy
fold                                 
1       0.0242       0.0332    0.1374
2       0.0242       0.0332    0.1374
3       0.0242       0.0332    0.1374
4       0.0242       0.0332    0.1374
5       0.0242       0.0332    0.1374

macro F1 per fold equals the P2 Dummy row of comparativa.csv


## Paired comparison against Practice 2

**The Practice 3 row of Challenge A.** Every value is copied from the saved outputs of R2's
notebook (P3.1). The spread is computed from the printed fold values, as in the Practice 2
rows. Before the row is added, the cell checks that the copied mean and spread agree with
the five copied fold values, to catch a typing mistake.

In [14]:
# Practice 3 row of Challenge A, copied from the saved outputs of R2's notebook
cyber_row = {
    "milestone": "P3",
    "challenge": "cyber",
    "family": "LogisticRegression",
    "route": "primary",
    "configuration": "multinomial, C=np.inf (no penalty; get_params() printed), "
                     "class_weight=None, solver=lbfgs, max_iter=5000; no hyperparameter search",
    "preprocessing": "one-hot on 3 nominal predictors with one reference level dropped "
                     "(proto=tcp, service=-, conn_state=SF), unseen level encoded as the "
                     "reference; log1p then StandardScaler on 8 numeric predictors; "
                     "fitted inside each fold",
    "features": "features.md: the 11 distributed source predictors (frozen basis)",
    "validation": "canonical StratifiedGroupKFold(5) on feature_group, "
                  "read from folds/cyber_folds.csv",
    "primary_metric_per_fold": "0.8407 0.8332 0.8453 0.8353 0.8361",
    "primary_metric": 0.8381,
    "primary_metric_spread": 0.0121,
    "supporting_metrics": "weighted F1 0.8533; accuracy 0.8546; per-class "
                          "precision/recall/F1/support and confusion in p3_cyber.ipynb",
    "fit_time_s": 7.6659,
    "interpretability": "odds ratios exp(beta_c - beta_r) of each class against the "
                        "reference class normal (P3.2)",
    "notes": "unpenalised linear model on the log-odds; mitm is the weakest class; coef_ has "
             "29 columns in fold 3 because service=gssapi is absent from its training rows",
}

# Check against typing mistakes: mean and spread must agree with the five fold values
copied_folds = np.array(cyber_row["primary_metric_per_fold"].split(), dtype=float)
mean_check, spread_check = mean_and_spread(copied_folds)
assert abs(mean_check - cyber_row["primary_metric"]) < 0.0001
assert abs(spread_check - cyber_row["primary_metric_spread"]) < 0.00001

# Add the row at the end of comparativa.csv (the Practice 2 rows are not touched)
append_comparativa_row(cyber_row)

added to comparativa.csv: P3 cyber LogisticRegression


In [15]:
# Paired differences, both rows read from comparativa.csv
new_scores = table_scores("P3", "cyber", "LogisticRegression")
old_scores = table_scores("P2", "cyber", "KNN")

# delta = P3 - P2 for macro F1, so a positive value means the logistic regression did better
cyber_deltas = paired_differences("cyber", new_scores, old_scores)

          P2     new   delta
fold                        
1     0.9436  0.8407 -0.1029
2     0.9473  0.8332 -0.1141
3     0.9490  0.8453 -0.1037
4     0.9428  0.8353 -0.1075
5     0.9451  0.8361 -0.1090
mean delta            -0.1074
spread of delta       0.0112   (max - min)
folds with delta > 0  0 of 5
spread of P2 row      0.0062
spread of new row     0.0121


**Reading of the differences, and which kind of comparison this is.** The logistic
regression has a lower macro F1 than the Practice 2 KNN on all five folds. The mean
difference is −0.1074, and the five differences stay within 0.0112 of each other, so the
drop has about the same size on every fold. It is also much larger than the fold-to-fold
spread of either row (0.0062 for KNN, 0.0121 for the logistic regression). This describes
these five folds, it is not a statistical test.

The two rows do not use the same representation: Practice 2 one-hot encodes every level of
the nominal predictors, while Practice 3 drops one reference level per predictor. So this is
a **pipeline comparison**, not an isolated comparison of model families: it says that this
pipeline did worse than that one on these folds, not that the model family alone explains
the difference.

# Challenge B — storage

> **Only in this draft.** As in Challenge A, in the final notebook these names come from
> R1's cells, so R5 does not copy the next cell.

In [16]:
# Only for this draft notebook: in p3_storage.ipynb these names come from R1's cells

# Development table and grouping metadata (the groups never enter the model)
dev = pd.read_csv("../datos/storage/storage_development.csv")
groups = pd.read_csv("../datos/storage/storage_groups.csv")

# The 9 source predictors: 4 nominal and 5 numeric
NOMINAL = ["load_type", "io_type", "raid", "device_type"]
NUMERIC = ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks"]
FEATURES = NOMINAL + NUMERIC

X = dev[FEATURES]                         # only the 9 source predictors
latency_ns = dev["latency_ns"].to_numpy() # latency in nanoseconds, as measured
z = np.log10(latency_ns)                  # modelling target, z = log10(latency_ns)

## Canonical folds and metrics (publish for R3)

The folds are read from `folds/storage_folds.csv`, the course file. They come from a
`GroupKFold(5)` on `config_group`, so the five replicates of a configuration always stay in
the same fold.

In [17]:
# The groups file must list the rows in the same order as the data table
assert np.array_equal(groups["row_id"].to_numpy(), dev["row_id"].to_numpy())

# Canonical folds of Challenge B: a list of five (train_idx, val_idx) pairs
folds = canonical_cv("storage", dev)

In [18]:
# Grouping rule: no config_group may be in training and in validation of the same fold
check_folds(folds, groups["config_group"])

# X must only hold the source predictors: never row_id, feature_group, config_group or fold_id
assert list(X.columns) == FEATURES

fold 1: 9750 training rows, 2440 validation rows, 0 groups in both parts
fold 2: 9750 training rows, 2440 validation rows, 0 groups in both parts
fold 3: 9750 training rows, 2440 validation rows, 0 groups in both parts
fold 4: 9755 training rows, 2435 validation rows, 0 groups in both parts
fold 5: 9755 training rows, 2435 validation rows, 0 groups in both parts
every row is validated exactly once


**Metric code for Challenge B** (the metric contract of Practice 2). As in Challenge A, the
model is fitted outside the function, for example with `cross_val_predict(model, X, z, cv=folds)`,
so the prediction of each row comes from a model that did not see it.

- MAE(z) (primary), RMSE(z) and R²(z) are computed from $z$ and $\hat z$.
- MAE in ns is computed only after turning each prediction back into nanoseconds with
  $10^{\hat z}$. Raising 10 to MAE(z) gives a multiplicative factor, never an error in ns.

In [19]:
# Metrics of Challenge B on each fold, from the out-of-fold predictions of z
def storage_fold_metrics(z, z_pred, latency_ns, folds):
    """MAE(z), RMSE(z), R2(z) and MAE in ns on the validation rows of each fold."""
    rows = []
    for number, (train_idx, val_idx) in enumerate(folds, start=1):
        # True and predicted z of the validation rows of this fold
        z_true_fold = z[val_idx]
        z_pred_fold = z_pred[val_idx]
        # Back to nanoseconds: we invert the predictions, never the MAE
        latency_pred_fold = 10 ** z_pred_fold
        # One dictionary per fold: three metrics on z, one metric in nanoseconds
        rows.append({"fold": number,
                     "mae_z": mean_absolute_error(z_true_fold, z_pred_fold),
                     "rmse_z": np.sqrt(mean_squared_error(z_true_fold, z_pred_fold)),
                     "r2_z": r2_score(z_true_fold, z_pred_fold),
                     "mae_ns": mean_absolute_error(latency_ns[val_idx], latency_pred_fold)})
    # Table with one row per fold
    return pd.DataFrame(rows).set_index("fold")

**Check: the folds and the metric code reproduce Practice 2.** We run again the Practice 2
baseline, `DummyRegressor(strategy="mean")` on $z$, and compare its MAE(z) per fold with the
P2 row of `comparativa.csv`.

In [20]:
# Out-of-fold predictions of the Practice 2 baseline (the mean of z) on the canonical folds
z_pred_dummy = cross_val_predict(DummyRegressor(strategy="mean"), X, z, cv=folds)

# Its metrics on each fold
dummy_metrics = storage_fold_metrics(z, z_pred_dummy, latency_ns, folds)
print(dummy_metrics.round(4).to_string())

# MAE(z) must equal the P2 Dummy row of comparativa.csv (rounded to 4 decimals there)
p2_dummy = table_scores("P2", "storage", "Dummy")
assert np.allclose(dummy_metrics["mae_z"], p2_dummy, atol=0.00005)
print("\nMAE(z) per fold equals the P2 Dummy row of comparativa.csv")

       mae_z  rmse_z    r2_z        mae_ns
fold                                      
1     0.6800  0.8445 -0.0002  2.839968e+07
2     0.6778  0.8436 -0.0002  2.854142e+07
3     0.6628  0.8331 -0.0001  2.742680e+07
4     0.6722  0.8269 -0.0001  2.407004e+07
5     0.6922  0.8517 -0.0000  2.674962e+07

MAE(z) per fold equals the P2 Dummy row of comparativa.csv


## Paired comparison against Practice 2

**The Practice 3 row of Challenge B.** Every value is copied from the saved outputs of R3's
notebook (P3.3). The row is the unregularised `LinearRegression`, the model recommended in
P3.4. As in Challenge A, the cell checks the copied mean and spread before adding the row.

In [21]:
# Practice 3 row of Challenge B, copied from the saved outputs of R3's notebook
storage_row = {
    "milestone": "P3",
    "challenge": "storage",
    "family": "LinearRegression",
    "route": "primary",
    "configuration": "ordinary least squares, no penalty; no hyperparameter search "
                     "(Ridge and Lasso compared in P3.4)",
    "preprocessing": "one-hot on the 4 nominal predictors with one reference level dropped "
                     "(load_type=random, io_type=write, raid=4+1, device_type=nvme), unseen "
                     "level encoded as the reference; StandardScaler on the 5 numeric ones; "
                     "fitted inside each fold; target z = log10(latency_ns)",
    "features": "features.md: the 9 distributed source predictors (frozen basis)",
    "validation": "canonical GroupKFold(5) on config_group, read from folds/storage_folds.csv",
    "primary_metric_per_fold": "0.2047 0.1937 0.1984 0.1953 0.1987",
    "primary_metric": 0.1981,
    "primary_metric_spread": 0.0110,
    "supporting_metrics": "RMSE(z) 0.2703; R2(z) 0.8964; MAE_ns 16289769 "
                          "(computed after inverting predictions)",
    "fit_time_s": 0.0298,   # mean fit time per fold printed by R3's notebook (approximate)
    "interpretability": "each coefficient is a factor 10^coef on latency: per standard "
                        "deviation of a numeric predictor, or per level against its reference",
    "notes": "additive model on z; worse than KNN on every fold; Ridge and Lasso gave "
             "no meaningful gain over it (P3.4); fit time is approximate and changes "
             "between runs",
}

# Check against typing mistakes: mean and spread must agree with the five fold values
copied_folds = np.array(storage_row["primary_metric_per_fold"].split(), dtype=float)
mean_check, spread_check = mean_and_spread(copied_folds)
assert abs(mean_check - storage_row["primary_metric"]) < 0.0001
assert abs(spread_check - storage_row["primary_metric_spread"]) < 0.00001

# Add the row at the end of comparativa.csv (the Practice 2 rows are not touched)
append_comparativa_row(storage_row)

added to comparativa.csv: P3 storage LinearRegression


In [22]:
# Paired differences, both rows read from comparativa.csv
new_scores = table_scores("P3", "storage", "LinearRegression")
old_scores = table_scores("P2", "storage", "KNN")

# delta = P2 - P3 for MAE(z), so a positive value means the linear model did better
storage_deltas = paired_differences("storage", new_scores, old_scores)

          P2     new   delta
fold                        
1     0.1816  0.2047 -0.0231
2     0.1870  0.1937 -0.0067
3     0.1778  0.1984 -0.0206
4     0.1775  0.1953 -0.0178
5     0.1762  0.1987 -0.0225
mean delta            -0.0181
spread of delta       0.0164   (max - min)
folds with delta > 0  0 of 5
spread of P2 row      0.0108
spread of new row     0.0110


**Reading of the differences, and which kind of comparison this is.** The linear regression
has a higher MAE(z) than the Practice 2 KNN on all five folds, so every difference is
negative. The mean difference is −0.0181. The size changes more from fold to fold than in
Challenge A: from −0.0231 to −0.0067 (spread 0.0164), against a fold-to-fold spread of about
0.011 for each of the two rows. So the direction is the same on every fold, but the size is
less stable. This describes these five folds; it is not a statistical test.

As in Challenge A, Practice 2 one-hot encodes every level and Practice 3 drops one reference
level per nominal predictor, so this is a **pipeline comparison**, not an isolated
comparison of model families.

# Comparison table

The Practice 3 rows are added in the paired-comparison part of each challenge, just before
their differences are computed, because the differences are read from the table. This last
cell shows the whole table so that we can check that the four Practice 2 rows are still the
first rows, unchanged, and that the new rows come after them.

Practice 2 and Practice 3 do not use the same representation of the nominal predictors:
Practice 2 one-hot encodes every level, Practice 3 drops one reference level per predictor.
The information basis is the same (the same source predictors, no derived feature), but the
representation is not, so every Practice 3 against Practice 2 difference is a **pipeline
comparison**, not an isolated comparison of model families.

In [23]:
# Read the whole table and show its main columns
table = pd.read_csv(COMPARISON_PATH)
print(table[["milestone", "challenge", "family", "route",
             "primary_metric", "primary_metric_spread"]].to_string())

# The table is only appended to: the Practice 2 rows are still the first four rows
assert list(table["milestone"].iloc[:4]) == ["P2", "P2", "P2", "P2"]

  milestone challenge              family    route  primary_metric  primary_metric_spread
0        P2     cyber               Dummy  primary          0.0242                 0.0000
1        P2   storage               Dummy  primary          0.6770                 0.0294
2        P2     cyber                 KNN  primary          0.9456                 0.0062
3        P2   storage                 KNN  primary          0.1800                 0.0109
4        P3     cyber  LogisticRegression  primary          0.8381                 0.0121
5        P3   storage    LinearRegression  primary          0.1981                 0.0110
